# Rolling 5-Year Retrained LightGBM + Baseline Yearly Comparison

This notebook is intentionally minimal. It only implements the rolling 5-year retrained LightGBM evaluation and the yearly baseline comparison requested for this task.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from IPython.display import display
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
universe = [
    "NVDA", "AMD", "INTC", "QCOM", "AVGO", "TXN", "MU", "ADI",
    "MRVL", "MCHP", "MPWR", "ON", "NXPI", "SWKS", "QRVO",
    "LSCC", "CRUS", "DIOD", "POWI", "ALGM",
    "AMAT", "LRCX", "KLAC", "TER", "ENTG", "ACLS", "AEHR",
    "ASML", "ASX", "UCTT", "FORM", "COHR",
    "TSM", "UMC", "STM", "GFS",
    "SNPS", "CDNS", "ANSS",
    "AAPL", "MSFT", "GOOGL", "META", "AMZN",
    "DELL", "HPQ", "SMCI", "HPE", "ARW", "AVT"
]

start_date = "2010-01-01"
end_date = "2024-12-31"

raw_data = yf.download(
    universe,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=True,
    group_by="ticker",
    threads=True,
)


[**********************78%************           ]  39 of 50 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
[*********************100%***********************]  50 of 50 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ANSS']: YFTzMissingError('possibly delisted; no timezone found')


In [ ]:
def make_price_panel(raw_data, requested_universe):
    frames = []
    valid_tickers = []

    for ticker in requested_universe:
        try:
            px = raw_data[ticker].copy()
        except Exception:
            continue

        if px.empty or "Close" not in px.columns or px["Close"].dropna().empty:
            continue

        keep_cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in px.columns]
        px = px[keep_cols].copy()
        px["code"] = ticker
        frames.append(px.reset_index().rename(columns={"Date": "date"}))
        valid_tickers.append(ticker)

    if not frames:
        raise ValueError("No valid tickers were downloaded. Check the data source or network connection.")

    df = pd.concat(frames, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["code", "date"]).reset_index(drop=True)

    price_cols = ["Open", "High", "Low", "Close"]
    df[price_cols] = df.groupby("code")[price_cols].ffill()
    df["Volume"] = df.groupby("code")["Volume"].transform(lambda x: x.fillna(0))
    df["Return"] = df.groupby("code")["Close"].pct_change()
    df = df.dropna(subset=price_cols + ["Return"]).reset_index(drop=True)
    return df, valid_tickers


df, valid_tickers = make_price_panel(raw_data, universe)
print(f"Valid tickers: {len(valid_tickers)} / {len(universe)}")
print(df.shape)
df.head()


Valid tickers: 49 / 50
(173996, 8)


Price,date,Open,High,Low,Close,Volume,code,Return
0,2010-01-05,6.424141,6.453777,6.383728,6.417555,601904800.0,AAPL,0.001729
1,2010-01-06,6.417559,6.443004,6.308893,6.315479,552160000.0,AAPL,-0.015906
2,2010-01-07,6.338826,6.346310,6.258000,6.303801,477131200.0,AAPL,-0.001849
3,2010-01-08,6.295419,6.346310,6.258300,6.345711,447610800.0,AAPL,0.006648
4,2010-01-11,6.370260,6.376247,6.240041,6.289734,462229600.0,AAPL,-0.008821


In [ ]:
def add_technical_features(df):
    df = df.sort_values(["code", "date"]).reset_index(drop=True).copy()
    g = df.groupby("code", group_keys=False)
    f = pd.DataFrame(index=df.index)

    f["open_1"] = g["Open"].shift(1)
    f["high_1"] = g["High"].shift(1)
    f["low_1"] = g["Low"].shift(1)
    f["close_1"] = g["Close"].shift(1)
    f["close_2"] = g["Close"].shift(2)
    f["volume_1"] = g["Volume"].shift(1)
    f["log_volume_1"] = np.log1p(f["volume_1"])

    f["overnight_gap_1"] = f["open_1"] / f["close_2"] - 1
    f["intraday_return_1"] = f["close_1"] / f["open_1"] - 1
    f["high_low_range_1"] = f["high_1"] / f["low_1"] - 1
    f["high_close_range_1"] = f["high_1"] / f["close_1"] - 1
    f["low_close_range_1"] = f["low_1"] / f["close_1"] - 1
    f["close_position_1"] = (f["close_1"] - f["low_1"]) / (f["high_1"] - f["low_1"]).replace(0, np.nan)
    f["candle_body_1"] = (f["close_1"] - f["open_1"]) / f["open_1"]
    f["candle_body_abs_1"] = f["candle_body_1"].abs()
    f["upper_shadow_1"] = (f["high_1"] - np.maximum(f["open_1"], f["close_1"])) / f["close_1"]
    f["lower_shadow_1"] = (np.minimum(f["open_1"], f["close_1"]) - f["low_1"]) / f["close_1"]

    for window in [1, 2, 5, 10, 20, 60, 120, 250]:
        f[f"return_{window}"] = g["Close"].transform(lambda x, w=window: (x / x.shift(w) - 1).shift(1))
    for window in [1, 5, 20, 60, 250]:
        f[f"log_return_{window}"] = g["Close"].transform(lambda x, w=window: np.log(x).diff(w).shift(1))

    f["momentum_20_minus_5"] = f["return_20"] - f["return_5"]
    f["momentum_60_minus_20"] = f["return_60"] - f["return_20"]
    f["momentum_250_minus_60"] = f["return_250"] - f["return_60"]

    for window in [5, 20, 60, 250]:
        f[f"ret_mean_{window}"] = g["Close"].transform(lambda x, w=window: x.pct_change().rolling(w).mean().shift(1))
        f[f"ret_std_{window}"] = g["Close"].transform(lambda x, w=window: x.pct_change().rolling(w).std().shift(1))
        f[f"sharpe_{window}"] = f[f"ret_mean_{window}"] / f[f"ret_std_{window}"]

    for window in [20, 60]:
        f[f"ret_skew_{window}"] = g["Close"].transform(lambda x, w=window: x.pct_change().rolling(w).skew().shift(1))
        f[f"ret_kurt_{window}"] = g["Close"].transform(lambda x, w=window: x.pct_change().rolling(w).kurt().shift(1))

    f["return_5_to_vol_20"] = f["return_5"] / f["ret_std_20"]
    f["return_20_to_vol_60"] = f["return_20"] / f["ret_std_60"]

    ma = {}
    for window in [5, 10, 20, 60, 120, 250]:
        ma[window] = g["Close"].transform(lambda x, w=window: x.rolling(w).mean().shift(1))
        f[f"close_to_ma_{window}"] = f["close_1"] / ma[window] - 1

    f["ma_5_to_20"] = ma[5] / ma[20] - 1
    f["ma_10_to_20"] = ma[10] / ma[20] - 1
    f["ma_20_to_60"] = ma[20] / ma[60] - 1
    f["ma_60_to_120"] = ma[60] / ma[120] - 1
    f["ma_60_to_250"] = ma[60] / ma[250] - 1
    f["ma_120_to_250"] = ma[120] / ma[250] - 1

    for window in [20, 60]:
        price_std = g["Close"].transform(lambda x, w=window: x.rolling(w).std().shift(1))
        f[f"bollinger_z_{window}"] = (f["close_1"] - ma[window]) / price_std
        f[f"bollinger_width_{window}"] = 4 * price_std / ma[window]

    for window in [20, 60, 250]:
        rolling_high = g["High"].transform(lambda x, w=window: x.rolling(w).max().shift(1))
        rolling_low = g["Low"].transform(lambda x, w=window: x.rolling(w).min().shift(1))
        f[f"dist_to_high_{window}"] = f["close_1"] / rolling_high - 1
        f[f"dist_to_low_{window}"] = f["close_1"] / rolling_low - 1
        f[f"range_position_{window}"] = (f["close_1"] - rolling_low) / (rolling_high - rolling_low).replace(0, np.nan)
        f[f"drawdown_{window}"] = f["close_1"] / rolling_high - 1

    prev_close = g["Close"].shift(1)
    true_range = pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    tr_df = pd.DataFrame({"code": df["code"], "true_range": true_range})
    tr_g = tr_df.groupby("code", group_keys=False)
    for window in [14, 20, 60]:
        f[f"natr_{window}"] = tr_g["true_range"].transform(lambda x, w=window: x.rolling(w).mean().shift(1)) / f["close_1"]

    def rsi(x, window):
        diff = x.diff()
        gain = diff.clip(lower=0).rolling(window).mean()
        loss = (-diff.clip(upper=0)).rolling(window).mean()
        return (100 - 100 / (1 + gain / loss)).shift(1)

    f["rsi_14"] = g["Close"].transform(lambda x: rsi(x, 14))
    f["rsi_28"] = g["Close"].transform(lambda x: rsi(x, 28))
    f["ema_12_to_26"] = g["Close"].transform(lambda x: (x.ewm(span=12, adjust=False).mean() / x.ewm(span=26, adjust=False).mean() - 1).shift(1))
    f["ema_20_to_60"] = g["Close"].transform(lambda x: (x.ewm(span=20, adjust=False).mean() / x.ewm(span=60, adjust=False).mean() - 1).shift(1))
    macd_signal = g["Close"].transform(lambda x: ((x.ewm(span=12, adjust=False).mean() - x.ewm(span=26, adjust=False).mean()) / x.ewm(span=26, adjust=False).mean()).ewm(span=9, adjust=False).mean().shift(1))
    f["macd_signal_9"] = macd_signal
    f["macd_hist"] = f["ema_12_to_26"] - f["macd_signal_9"]

    for window in [1, 5, 20]:
        f[f"volume_log_change_{window}"] = g["Volume"].transform(lambda x, w=window: np.log1p(x).diff(w).shift(1))

    log_volume = np.log1p(df["Volume"])
    lv_df = pd.DataFrame({"code": df["code"], "log_volume": log_volume})
    lv_g = lv_df.groupby("code", group_keys=False)
    for window in [20, 60]:
        lv_mean = lv_g["log_volume"].transform(lambda x, w=window: x.rolling(w).mean().shift(1))
        lv_std = lv_g["log_volume"].transform(lambda x, w=window: x.rolling(w).std().shift(1))
        f[f"relative_log_volume_{window}"] = f["log_volume_1"] - lv_mean
        f[f"volume_zscore_{window}"] = (f["log_volume_1"] - lv_mean) / lv_std
        f[f"volume_rank_{window}"] = lv_g["log_volume"].transform(lambda x, w=window: x.rolling(w).rank(pct=True).shift(1))

    dollar_volume = df["Close"] * df["Volume"]
    dv_df = pd.DataFrame({"code": df["code"], "dollar_volume": dollar_volume})
    dv_g = dv_df.groupby("code", group_keys=False)
    for window in [1, 5, 20]:
        f[f"dollar_volume_log_change_{window}"] = dv_g["dollar_volume"].transform(lambda x, w=window: np.log1p(x).diff(w).shift(1))

    cs_base = pd.concat([df[["date"]], f], axis=1)
    for col in ["return_1", "return_5", "return_20", "return_60", "ret_std_20", "ret_std_60", "volume_zscore_20", "close_to_ma_20", "range_position_60"]:
        f[f"{col}_cs_rank"] = cs_base.groupby("date")[col].rank(pct=True)

    f["day_of_week"] = df["date"].dt.dayofweek
    f["month"] = df["date"].dt.month
    f["quarter"] = df["date"].dt.quarter
    f["is_month_start"] = df["date"].dt.is_month_start.astype(int)
    f["is_month_end"] = df["date"].dt.is_month_end.astype(int)
    f["is_quarter_start"] = df["date"].dt.is_quarter_start.astype(int)
    f["is_quarter_end"] = df["date"].dt.is_quarter_end.astype(int)

    out = pd.concat([df, f], axis=1)
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


df = add_technical_features(df)
print(df.shape)
df.head()


/tmp/ipykernel_15594/1034038970.py:117: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  f[f"{col}_cs_rank"] = cs_base.groupby("date")[col].rank(pct=True)
/tmp/ipykernel_15594/1034038970.py:117: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  f[f"{col}_cs_rank"] = cs_base.groupby("date")[col].rank(pct=True)
/tmp/ipykernel_15594/1034038970.py:117: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using 

(173996, 124)


,date,Open,High,Low,Close,Volume,code,Return,open_1,high_1,low_1,close_1,close_2,volume_1,log_volume_1,overnight_gap_1,intraday_return_1,high_low_range_1,high_close_range_1,low_close_range_1,close_position_1,candle_body_1,candle_body_abs_1,upper_shadow_1,lower_shadow_1,return_1,return_2,return_5,return_10,return_20,return_60,return_120,return_250,log_return_1,log_return_5,log_return_20,log_return_60,log_return_250,momentum_20_minus_5,momentum_60_minus_20,momentum_250_minus_60,ret_mean_5,ret_std_5,sharpe_5,ret_mean_20,ret_std_20,sharpe_20,ret_mean_60,ret_std_60,sharpe_60,ret_mean_250,ret_std_250,sharpe_250,ret_skew_20,ret_kurt_20,ret_skew_60,ret_kurt_60,return_5_to_vol_20,return_20_to_vol_60,close_to_ma_5,close_to_ma_10,close_to_ma_20,close_to_ma_60,close_to_ma_120,close_to_ma_250,ma_5_to_20,ma_10_to_20,ma_20_to_60,ma_60_to_120,ma_60_to_250,ma_120_to_250,bollinger_z_20,bollinger_width_20,bollinger_z_60,bollinger_width_60,dist_to_high_20,dist_to_low_20,range_position_20,drawdown_20,dist_to_high_60,dist_to_low_60,range_position_60,drawdown_60,dist_to_high_250,dist_to_low_250,range_position_250,drawdown_250,natr_14,natr_20,natr_60,rsi_14,rsi_28,ema_12_to_26,ema_20_to_60,macd_signal_9,macd_hist,volume_log_change_1,volume_log_change_5,volume_log_change_20,relative_log_volume_20,volume_zscore_20,volume_rank_20,relative_log_volume_60,volume_zscore_60,volume_rank_60,dollar_volume_log_change_1,dollar_volume_log_change_5,dollar_volume_log_change_20,return_1_cs_rank,return_5_cs_rank,return_20_cs_rank,return_60_cs_rank,ret_std_20_cs_rank,ret_std_60_cs_rank,volume_zscore_20_cs_rank,close_to_ma_20_cs_rank,range_position_60_cs_rank,day_of_week,month,quarter,is_month_start,is_month_end,is_quarter_start,is_quarter_end
0,2010-01-05,6.424141,6.453777,6.383728,6.417555,601904800.0,AAPL,0.001729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,0,0,0,0
1,2010-01-06,6.417559,6.443004,6.308893,6.315479,552160000.0,AAPL,-0.015906,6.424141,6.453777,6.383728,6.417555,NaN,601904800.0,20.215610,NaN,-0.001025,0.010973,0.005644,-0.005271,0.482909,-0.001025,0.001025,0.004618,0.005271,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,1,1,0,0,0,0
2,2010-01-07,6.338826,6.346310,6.258000,6.303801,477131200.0,AAPL,-0.001849,6.417559,6.443004,6.308893,6.315479,6.417555,552160000.0,20.129348,5.045902e-07,-0.015906,0.021257,0.020192,-0.001043,0.049108,-0.015906,0.015906,0.004029,0.001043,-0.015906,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.016034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.001270,-0.000994,-0.000254,-0.001016,-0.086261,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.102295,NaN,NaN,0.166667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,1,1,0,0,0,0
3,2010-01-08,6.295419,6.346310,6.258300,6.345711,447610800.0,AAPL,0.006648,6.338826,6.346310,6.258000,6.303801,6.315479,477131200.0,19.983302,3.696807e-03,-0.005525,0.014112,0.006743,-0.007266,0.518640,-0.005525,0.005525,0.001187,0.007266,-0.001849,-0.017725,NaN,NaN,NaN,NaN,NaN,NaN,-0.001851,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [ ]:
df["target_5d"] = df.groupby("code")["Close"].shift(-5) / df["Close"] - 1

base_cols = ["date", "code", "Open", "High", "Low", "Close", "Volume", "Return", "target_5d"]
feature_cols = [c for c in df.columns if c not in base_cols]

df_model = df.dropna(subset=feature_cols + ["target_5d"]).copy().reset_index(drop=True)


def create_time_split(
    df_model,
    train_start,
    train_end,
    val_start,
    val_end,
    test_start,
    test_end,
    date_col="date",
):
    df_sorted = df_model.sort_values([date_col, "code"]).copy()
    split = {
        "train": df_sorted[df_sorted[date_col].between(train_start, train_end)].copy(),
        "validation": df_sorted[df_sorted[date_col].between(val_start, val_end)].copy(),
        "test": df_sorted[df_sorted[date_col].between(test_start, test_end)].copy(),
    }
    split_dates = {
        "train": (train_start, train_end),
        "validation": (val_start, val_end),
        "test": (test_start, test_end),
    }
    return split, split_dates


def prepare_walk_forward_model_data(df_source):
    wf = df_source.copy()
    wf["target_5d_raw"] = wf.groupby("code")["Close"].shift(-5) / wf["Close"] - 1
    wf["target_5d_cs"] = wf["target_5d_raw"] - wf.groupby("date")["target_5d_raw"].transform("mean")

    wf_base_cols = [
        "date", "code", "Open", "High", "Low", "Close", "Volume", "Return",
        "target_5d", "target_5d_raw", "target_5d_cs",
    ]
    wf_feature_cols = [c for c in wf.columns if c not in wf_base_cols]
    wf = wf.dropna(subset=wf_feature_cols + ["target_5d_raw", "target_5d_cs"]).copy().reset_index(drop=True)
    return wf, wf_feature_cols


walk_forward_model_df, walk_forward_feature_cols = prepare_walk_forward_model_data(df)
print(df_model.shape, walk_forward_model_df.shape, len(walk_forward_feature_cols))


(161322, 125) (161322, 127) 116


In [ ]:
def get_rebalance_dates(df_eval, every_n_days=5):
    dates = np.array(sorted(pd.to_datetime(df_eval["date"].unique())))
    return dates[::every_n_days]


def build_long_short_weights(
    df_eval,
    signal_col,
    date_col="date",
    code_col="code",
    top_frac=0.20,
    rebalance_dates=None,
    min_names=10,
):
    if rebalance_dates is None:
        rebalance_dates = get_rebalance_dates(df_eval, every_n_days=5)

    weight_frames = []
    for date in rebalance_dates:
        cross_section = df_eval[df_eval[date_col] == date][[date_col, code_col, signal_col]].dropna().copy()
        n = len(cross_section)
        if n < min_names:
            continue

        n_bucket = max(1, int(np.floor(n * top_frac)))
        cross_section = cross_section.sort_values(signal_col, ascending=False)
        longs = cross_section.head(n_bucket).copy()
        shorts = cross_section.tail(n_bucket).copy()

        weights = pd.concat([longs, shorts], ignore_index=True)
        weights["weight"] = 0.0
        weights.loc[weights[code_col].isin(longs[code_col]), "weight"] = 0.5 / len(longs)
        weights.loc[weights[code_col].isin(shorts[code_col]), "weight"] = -0.5 / len(shorts)
        weight_frames.append(weights[[date_col, code_col, "weight"]])

    if not weight_frames:
        return pd.DataFrame(columns=[date_col, code_col, "weight"])

    return pd.concat(weight_frames, ignore_index=True)


def build_equal_weight_long_only_weights(df_eval, rebalance_dates=None, date_col="date", code_col="code"):
    if rebalance_dates is None:
        rebalance_dates = get_rebalance_dates(df_eval, every_n_days=5)

    frames = []
    for date in rebalance_dates:
        names = sorted(df_eval.loc[df_eval[date_col] == date, code_col].dropna().unique())
        if len(names) == 0:
            continue
        frames.append(pd.DataFrame({date_col: date, code_col: names, "weight": 1.0 / len(names)}))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=[date_col, code_col, "weight"])


def build_random_long_short_weights(df_eval, rebalance_dates=None, top_frac=0.20, seed=42, date_col="date", code_col="code", min_names=10):
    rng = np.random.default_rng(seed)
    if rebalance_dates is None:
        rebalance_dates = get_rebalance_dates(df_eval, every_n_days=5)

    frames = []
    for date in rebalance_dates:
        names = np.array(sorted(df_eval.loc[df_eval[date_col] == date, code_col].dropna().unique()))
        n = len(names)
        if n < min_names:
            continue
        n_bucket = max(1, int(np.floor(n * top_frac)))
        chosen = rng.choice(names, size=2 * n_bucket, replace=False)
        longs = chosen[:n_bucket]
        shorts = chosen[n_bucket:]
        w = pd.DataFrame({date_col: date, code_col: np.concatenate([longs, shorts])})
        w["weight"] = np.concatenate([np.repeat(0.5 / n_bucket, n_bucket), np.repeat(-0.5 / n_bucket, n_bucket)])
        frames.append(w)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=[date_col, code_col, "weight"])


def long_short_backtest(
    df_eval,
    weights,
    target_col="target_5d",
    transaction_cost=0.0005,
    date_col="date",
    code_col="code",
):
    df_bt = weights.merge(df_eval[[date_col, code_col, target_col]], on=[date_col, code_col], how="left")
    df_bt = df_bt.dropna(subset=[target_col, "weight"]).copy()

    rows = []
    prev_weights = pd.Series(dtype=float)

    for date, part in df_bt.groupby(date_col, sort=True):
        current_weights = part.set_index(code_col)["weight"].astype(float)
        all_names = current_weights.index.union(prev_weights.index)
        turnover = (current_weights.reindex(all_names, fill_value=0.0) - prev_weights.reindex(all_names, fill_value=0.0)).abs().sum()
        gross_return = (part["weight"] * part[target_col]).sum()
        cost = transaction_cost * turnover
        net_return = gross_return - cost

        rows.append({
            "date": pd.Timestamp(date),
            "gross_return": gross_return,
            "turnover": turnover,
            "cost": cost,
            "net_return": net_return,
            "n_positions": len(current_weights),
            "net_exposure": current_weights.sum(),
            "gross_exposure": current_weights.abs().sum(),
        })
        prev_weights = current_weights

    result = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    if result.empty:
        return result

    result["gross_cum_return"] = (1 + result["gross_return"]).cumprod() - 1
    result["net_cum_return"] = (1 + result["net_return"]).cumprod() - 1
    result["gross_equity"] = (1 + result["gross_return"]).cumprod()
    result["net_equity"] = (1 + result["net_return"]).cumprod()
    result["net_drawdown"] = result["net_equity"] / result["net_equity"].cummax() - 1
    return result


def backtest_metrics(backtest_result, return_col="net_return", periods_per_year=252 / 5):
    r = backtest_result[return_col].dropna()
    if r.empty:
        return {
            "Cumulative Return": np.nan,
            "Annualized Return": np.nan,
            "Annualized Volatility": np.nan,
            "Sharpe Ratio": np.nan,
            "Max Drawdown": np.nan,
            "Win Rate": np.nan,
            "Average Turnover": np.nan,
            "Number of Rebalance Periods": 0,
        }

    equity = (1 + r).cumprod()
    cumulative_return = equity.iloc[-1] - 1
    ann_return = equity.iloc[-1] ** (periods_per_year / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol and ann_vol > 0 else np.nan
    drawdown = equity / equity.cummax() - 1

    return {
        "Cumulative Return": cumulative_return,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": drawdown.min(),
        "Win Rate": (r > 0).mean(),
        "Average Turnover": backtest_result["turnover"].mean() if "turnover" in backtest_result else np.nan,
        "Number of Rebalance Periods": len(r),
    }


def evaluate_prediction(y_true, y_pred):
    y_true = pd.Series(y_true).astype(float).reset_index(drop=True)
    y_pred = pd.Series(y_pred).astype(float).reset_index(drop=True)
    valid = y_true.notna() & y_pred.notna()
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    pearson = y_true.corr(y_pred, method="pearson")
    spearman = y_true.corr(y_pred, method="spearman")
    direction_accuracy = (np.sign(y_true) == np.sign(y_pred)).mean()

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Pearson Corr": pearson,
        "Spearman Rank IC": spearman,
        "Direction Accuracy": direction_accuracy,
    }


def daily_rank_ic_summary(df_eval, signal_col, target_col="target_5d_cs"):
    daily_values = []
    for _, part in df_eval[["date", target_col, signal_col]].dropna().groupby("date", sort=True):
        if part[target_col].nunique() > 1 and part[signal_col].nunique() > 1:
            daily_values.append(part[target_col].corr(part[signal_col], method="spearman"))
    daily_ic = pd.Series(daily_values, dtype=float).dropna()
    return daily_ic.mean(), daily_ic.std(), len(daily_ic)


In [ ]:
def prepare_robust_simple_alpha_signals(df_part):
    out = df_part.copy()
    signal_cols = []

    for col in [
        "return_5", "return_10", "return_20", "return_60", "return_120",
        "close_to_ma_20", "close_to_ma_60",
    ]:
        if col in out.columns:
            signal_cols.append(col)

    if "return_20" in out.columns and "ret_std_20" in out.columns:
        out["return_20_to_ret_std_20"] = out["return_20"] / out["ret_std_20"].replace(0, np.nan)
        signal_cols.append("return_20_to_ret_std_20")

    if "return_60" in out.columns and "ret_std_60" in out.columns:
        out["return_60_to_ret_std_60"] = out["return_60"] / out["ret_std_60"].replace(0, np.nan)
        signal_cols.append("return_60_to_ret_std_60")

    if "rsi_14" in out.columns:
        out["rsi_reversal"] = -out["rsi_14"]
        signal_cols.append("rsi_reversal")

    if "bollinger_z_20" in out.columns:
        signal_cols.append("bollinger_z_20")
        out["neg_bollinger_z_20"] = -out["bollinger_z_20"]
        signal_cols.append("neg_bollinger_z_20")

    return out.replace([np.inf, -np.inf], np.nan), signal_cols


def build_robust_simple_weights(
    df_eval,
    signal_col,
    top_frac=0.20,
    rebalance_days=5,
    weighting_method="equal",
    vol_col="ret_std_20",
    date_col="date",
    code_col="code",
    min_names=10,
):
    rebalance_dates = get_rebalance_dates(df_eval, every_n_days=rebalance_days)
    weight_frames = []

    for date in rebalance_dates:
        cols = [date_col, code_col, signal_col]
        if vol_col in df_eval.columns:
            cols.append(vol_col)
        cross_section = df_eval.loc[df_eval[date_col] == date, cols].dropna(subset=[signal_col]).copy()
        if len(cross_section) < min_names:
            continue

        n_bucket = max(1, int(np.floor(len(cross_section) * top_frac)))
        cross_section = cross_section.sort_values(signal_col, ascending=False)
        longs = cross_section.head(n_bucket).copy()
        shorts = cross_section.tail(n_bucket).copy()

        def allocate(side, total_exposure):
            if weighting_method == "inverse_vol" and vol_col in side.columns:
                vol = side[vol_col].astype(float).replace(0, np.nan).abs()
                median_vol = vol.median()
                if pd.isna(median_vol) or median_vol <= 0:
                    median_vol = 1.0
                vol = vol.fillna(median_vol)
                inv_vol = 1.0 / vol
                if inv_vol.sum() > 0:
                    raw = inv_vol / inv_vol.sum()
                else:
                    raw = pd.Series(1.0 / len(side), index=side.index)
            else:
                raw = pd.Series(1.0 / len(side), index=side.index)
            return total_exposure * raw

        longs["weight"] = allocate(longs, 0.5)
        shorts["weight"] = allocate(shorts, -0.5)
        weight_frames.append(pd.concat([longs, shorts], ignore_index=True)[[date_col, code_col, "weight"]])

    if not weight_frames:
        return pd.DataFrame(columns=[date_col, code_col, "weight"])
    return pd.concat(weight_frames, ignore_index=True)


def robust_simple_backtest(
    df_eval,
    signal_col,
    direction=1,
    top_frac=0.20,
    rebalance_days=5,
    weighting_method="equal",
    transaction_cost=0.0005,
):
    work = df_eval.copy()
    directed_col = "_robust_directed_signal"
    work[directed_col] = direction * work[signal_col]
    work = work.dropna(subset=[directed_col, "target_5d_raw"])
    weights = build_robust_simple_weights(
        work,
        directed_col,
        top_frac=top_frac,
        rebalance_days=rebalance_days,
        weighting_method=weighting_method,
    )
    bt = long_short_backtest(
        work,
        weights,
        target_col="target_5d_raw",
        transaction_cost=transaction_cost,
    )
    gross_metrics = backtest_metrics(bt, return_col="gross_return", periods_per_year=252 / rebalance_days)
    net_metrics = backtest_metrics(bt, return_col="net_return", periods_per_year=252 / rebalance_days)
    return bt, gross_metrics, net_metrics


def run_robust_simple_alpha_search(df_val_eval, signal_cols):
    rows = []
    for signal_col in signal_cols:
        for direction in [1, -1]:
            for top_frac in [0.10, 0.15, 0.20, 0.30]:
                for rebalance_days in [5, 10, 20]:
                    for weighting_method in ["equal", "inverse_vol"]:
                        _, _, net = robust_simple_backtest(
                            df_val_eval,
                            signal_col,
                            direction=direction,
                            top_frac=top_frac,
                            rebalance_days=rebalance_days,
                            weighting_method=weighting_method,
                            transaction_cost=0.0005,
                        )
                        rows.append({
                            "Signal": signal_col,
                            "Direction": direction,
                            "top_frac": top_frac,
                            "rebalance_days": rebalance_days,
                            "weighting_method": weighting_method,
                            "Validation Net Cumulative Return": net["Cumulative Return"],
                            "Validation Net Sharpe": net["Sharpe Ratio"],
                            "Validation Max Drawdown": net["Max Drawdown"],
                            "Validation Average Turnover": net["Average Turnover"],
                            "Validation Win Rate": net["Win Rate"],
                        })

    results = pd.DataFrame(rows)
    results = results.sort_values(
        [
            "Validation Net Sharpe",
            "Validation Net Cumulative Return",
            "Validation Max Drawdown",
            "Validation Average Turnover",
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    return results


In [ ]:
if not HAS_LIGHTGBM:
    raise ImportError("LightGBM is required for this notebook.")

rolling5y_lgbm_candidates = [
    {
        "Candidate": "conservative shallow",
        "params": {
            "n_estimators": 100,
            "learning_rate": 0.03,
            "max_depth": 3,
            "num_leaves": 15,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 1.0,
            "reg_lambda": 10.0,
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": -1,
        },
    },
    {
        "Candidate": "medium",
        "params": {
            "n_estimators": 200,
            "learning_rate": 0.03,
            "max_depth": 4,
            "num_leaves": 31,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.5,
            "reg_lambda": 5.0,
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": -1,
        },
    },
    {
        "Candidate": "slightly stronger",
        "params": {
            "n_estimators": 300,
            "learning_rate": 0.02,
            "max_depth": 4,
            "num_leaves": 31,
            "subsample": 0.7,
            "colsample_bytree": 0.7,
            "reg_alpha": 1.0,
            "reg_lambda": 10.0,
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": -1,
        },
    },
]

rolling5y_fold_specs = [
    ("Fold 1", "2010-01-01", "2014-12-31", "2015-01-01", "2015-12-31", "2016-01-01", "2016-12-31"),
    ("Fold 2", "2011-01-01", "2015-12-31", "2016-01-01", "2016-12-31", "2017-01-01", "2017-12-31"),
    ("Fold 3", "2012-01-01", "2016-12-31", "2017-01-01", "2017-12-31", "2018-01-01", "2018-12-31"),
    ("Fold 4", "2013-01-01", "2017-12-31", "2018-01-01", "2018-12-31", "2019-01-01", "2019-12-31"),
    ("Fold 5", "2014-01-01", "2018-12-31", "2019-01-01", "2019-12-31", "2020-01-01", "2020-12-31"),
    ("Fold 6", "2015-01-01", "2019-12-31", "2020-01-01", "2020-12-31", "2021-01-01", "2021-12-31"),
    ("Fold 7", "2016-01-01", "2020-12-31", "2021-01-01", "2021-12-31", "2022-01-01", "2022-12-31"),
    ("Fold 8", "2017-01-01", "2021-12-31", "2022-01-01", "2022-12-31", "2023-01-01", "2023-12-31"),
    ("Fold 9", "2018-01-01", "2022-12-31", "2023-01-01", "2023-12-31", "2024-01-01", "2024-12-31"),
]


def get_rolling5y_source_data():
    if "walk_forward_model_df" in globals() and "walk_forward_feature_cols" in globals():
        source_df = walk_forward_model_df.copy()
        feature_cols = list(walk_forward_feature_cols)
    elif "df_model" in globals():
        source_df = df_model.copy()
        source_df["target_5d_raw"] = source_df.groupby("code")["Close"].shift(-5) / source_df["Close"] - 1
        source_df["target_5d_cs"] = source_df["target_5d_raw"] - source_df.groupby("date")["target_5d_raw"].transform("mean")
        base_cols = [
            "date", "code", "Open", "High", "Low", "Close", "Volume", "Return",
            "target_5d", "target_5d_raw", "target_5d_cs",
        ]
        feature_cols = [c for c in source_df.columns if c not in base_cols]
        source_df = source_df.dropna(subset=feature_cols + ["target_5d_raw", "target_5d_cs"]).copy().reset_index(drop=True)
    elif "df" in globals():
        source_df = df.copy()
        source_df["target_5d_raw"] = source_df.groupby("code")["Close"].shift(-5) / source_df["Close"] - 1
        source_df["target_5d_cs"] = source_df["target_5d_raw"] - source_df.groupby("date")["target_5d_raw"].transform("mean")
        base_cols = [
            "date", "code", "Open", "High", "Low", "Close", "Volume", "Return",
            "target_5d", "target_5d_raw", "target_5d_cs",
        ]
        feature_cols = [c for c in source_df.columns if c not in base_cols]
        source_df = source_df.dropna(subset=feature_cols + ["target_5d_raw", "target_5d_cs"]).copy().reset_index(drop=True)
    else:
        raise ValueError("No usable source dataframe found. Expected walk_forward_model_df, df_model, or df.")

    source_df["date"] = pd.to_datetime(source_df["date"])
    source_df = source_df.sort_values(["date", "code"]).reset_index(drop=True)
    return source_df, feature_cols


def rolling5y_build_model(params):
    return lgb.LGBMRegressor(objective="regression", **params)


def rolling5y_candidate_validation_row(candidate, X_train, y_train, X_val, y_val, val_df):
    model = rolling5y_build_model(candidate["params"])
    model.fit(X_train, y_train)
    raw_pred = model.predict(X_val)
    rows = []

    for direction in [1, -1]:
        pred = direction * raw_pred
        tmp = val_df[["date", "code", "target_5d_cs", "target_5d_raw"]].copy()
        tmp["rolling5y_signal"] = pred
        pred_metrics = evaluate_prediction(y_val, pred)
        daily_ic_mean, _, _ = daily_rank_ic_summary(tmp, "rolling5y_signal", target_col="target_5d_cs")
        rebalance_dates = get_rebalance_dates(tmp, every_n_days=5)
        weights = build_long_short_weights(tmp, "rolling5y_signal", rebalance_dates=rebalance_dates)
        bt = long_short_backtest(tmp, weights, target_col="target_5d_raw", transaction_cost=0.0005)
        net_metrics = backtest_metrics(bt, return_col="net_return")
        rows.append({
            "Candidate": candidate["Candidate"],
            "Params": candidate["params"],
            "Direction": direction,
            "Validation Net Sharpe": net_metrics["Sharpe Ratio"],
            "Validation Daily Rank IC Mean": daily_ic_mean,
            "Validation Net Cumulative Return": net_metrics["Cumulative Return"],
            "Validation Max Drawdown": net_metrics["Max Drawdown"],
            "Validation Average Turnover": net_metrics["Average Turnover"],
            "Validation Number of Rebalance Periods": net_metrics["Number of Rebalance Periods"],
            "Validation Spearman Rank IC": pred_metrics["Spearman Rank IC"],
        })
    return pd.DataFrame(rows)


def rolling5y_select_best_candidate(candidate_results_df, sharpe_tolerance=0.02):
    work = candidate_results_df.copy()
    sharpe = pd.to_numeric(work["Validation Net Sharpe"], errors="coerce")
    best_sharpe = sharpe.max()
    if pd.isna(best_sharpe):
        shortlist = work.copy()
    else:
        shortlist = work.loc[sharpe >= best_sharpe - sharpe_tolerance].copy()

    shortlist["Validation Average Turnover Sort"] = pd.to_numeric(shortlist["Validation Average Turnover"], errors="coerce").fillna(np.inf)
    shortlist["Validation Max Drawdown Abs Sort"] = pd.to_numeric(shortlist["Validation Max Drawdown"], errors="coerce").abs().fillna(np.inf)
    shortlist["Validation Net Sharpe Sort"] = pd.to_numeric(shortlist["Validation Net Sharpe"], errors="coerce").fillna(-np.inf)
    shortlist["Validation Net Cumulative Return Sort"] = pd.to_numeric(shortlist["Validation Net Cumulative Return"], errors="coerce").fillna(-np.inf)

    selected = shortlist.sort_values(
        [
            "Validation Net Sharpe Sort",
            "Validation Average Turnover Sort",
            "Validation Max Drawdown Abs Sort",
            "Validation Net Cumulative Return Sort",
        ],
        ascending=[False, True, True, False],
    ).iloc[0]
    return selected


def rolling5y_metric_row(fold_name, strategy, test_period, bt, periods_per_year=252 / 5):
    metrics = backtest_metrics(bt, return_col="net_return", periods_per_year=periods_per_year)
    return {
        "Fold": fold_name,
        "Strategy": strategy,
        "Test Period": test_period,
        "Test Net Cumulative Return": metrics["Cumulative Return"],
        "Test Sharpe": metrics["Sharpe Ratio"],
        "Test Max Drawdown": metrics["Max Drawdown"],
        "Test Average Turnover": metrics["Average Turnover"],
        "Number of Rebalance Periods": metrics["Number of Rebalance Periods"],
    }


def run_rolling5y_lightgbm_fold(fold_name, train_start, train_end, val_start, val_end, test_start, test_end, source_df, feature_cols):
    split, _ = create_time_split(
        source_df,
        train_start=train_start,
        train_end=train_end,
        val_start=val_start,
        val_end=val_end,
        test_start=test_start,
        test_end=test_end,
    )
    train_df = split["train"].copy()
    val_df = split["validation"].copy()
    test_df = split["test"].copy()

    X_train = train_df[feature_cols]
    y_train = train_df["target_5d_cs"]
    X_val = val_df[feature_cols]
    y_val = val_df["target_5d_cs"]
    X_test = test_df[feature_cols]
    y_test = test_df["target_5d_cs"]

    candidate_frames = []
    for candidate in rolling5y_lgbm_candidates:
        candidate_frames.append(rolling5y_candidate_validation_row(candidate, X_train, y_train, X_val, y_val, val_df))
    candidate_results_df = pd.concat(candidate_frames, ignore_index=True)
    selected = rolling5y_select_best_candidate(candidate_results_df)

    final_model = rolling5y_build_model(selected["Params"])
    X_train_val = pd.concat([X_train, X_val], axis=0)
    y_train_val = pd.concat([y_train, y_val], axis=0)
    final_model.fit(X_train_val, y_train_val)

    test_signal = int(selected["Direction"]) * final_model.predict(X_test)
    test_pred_metrics = evaluate_prediction(y_test, test_signal)

    test_eval = test_df[["date", "code", "target_5d_cs", "target_5d_raw"]].copy()
    test_eval["rolling5y_lgbm_signal"] = test_signal
    test_daily_ic_mean, _, _ = daily_rank_ic_summary(test_eval, "rolling5y_lgbm_signal", target_col="target_5d_cs")
    test_rebalance_dates = get_rebalance_dates(test_eval, every_n_days=5)
    test_weights = build_long_short_weights(test_eval, "rolling5y_lgbm_signal", rebalance_dates=test_rebalance_dates)
    test_bt = long_short_backtest(test_eval, test_weights, target_col="target_5d_raw", transaction_cost=0.0005)
    test_metrics = backtest_metrics(test_bt, return_col="net_return")

    row = {
        "Fold": fold_name,
        "Train Period": f"{train_start} to {train_end}",
        "Validation Period": f"{val_start} to {val_end}",
        "Test Period": f"{test_start} to {test_end}",
        "Number of Train Rows": len(train_df),
        "Number of Validation Rows": len(val_df),
        "Number of Test Rows": len(test_df),
        "Number of Train Stocks": train_df["code"].nunique(),
        "Number of Validation Stocks": val_df["code"].nunique(),
        "Number of Test Stocks": test_df["code"].nunique(),
        "Selected Candidate": selected["Candidate"],
        "Selected Params": selected["Params"],
        "Selected Direction": int(selected["Direction"]),
        "Validation Net Sharpe": selected["Validation Net Sharpe"],
        "Validation Daily Rank IC Mean": selected["Validation Daily Rank IC Mean"],
        "Test Spearman Rank IC": test_pred_metrics["Spearman Rank IC"],
        "Test Daily Rank IC Mean": test_daily_ic_mean,
        "Test Net Cumulative Return": test_metrics["Cumulative Return"],
        "Test Sharpe": test_metrics["Sharpe Ratio"],
        "Test Max Drawdown": test_metrics["Max Drawdown"],
        "Test Average Turnover": test_metrics["Average Turnover"],
        "Number of Rebalance Periods": test_metrics["Number of Rebalance Periods"],
    }

    print(
        f"{fold_name} | train {train_start} to {train_end} rows={len(train_df):,} stocks={train_df['code'].nunique()} | "
        f"validation {val_start} to {val_end} rows={len(val_df):,} stocks={val_df['code'].nunique()} | "
        f"test {test_start} to {test_end} rows={len(test_df):,} stocks={test_df['code'].nunique()}"
    )
    print(
        f"{fold_name} selected candidate={selected['Candidate']} direction={int(selected['Direction'])} "
        f"test_return={test_metrics['Cumulative Return']:.4f} test_sharpe={test_metrics['Sharpe Ratio']:.4f}"
    )

    return row, test_bt, split


rolling5y_source_df, rolling5y_feature_cols = get_rolling5y_source_data()
rolling5y_walk_forward_rows = []
rolling5y_walk_forward_backtests = {}
rolling5y_walk_forward_baseline_rows = []

for spec in rolling5y_fold_specs:
    fold_name, train_start, train_end, val_start, val_end, test_start, test_end = spec
    result_row, lgbm_bt, split = run_rolling5y_lightgbm_fold(
        fold_name, train_start, train_end, val_start, val_end, test_start, test_end, rolling5y_source_df, rolling5y_feature_cols
    )
    rolling5y_walk_forward_rows.append(result_row)
    rolling5y_walk_forward_backtests[fold_name] = lgbm_bt

    test_df = split["test"].copy()
    val_df = split["validation"].copy()
    test_period = f"{test_start} to {test_end}"
    test_rebalance_dates = get_rebalance_dates(test_df, every_n_days=5)

    rolling5y_walk_forward_baseline_rows.append(
        rolling5y_metric_row(fold_name, "LightGBM long-short", test_period, lgbm_bt, periods_per_year=252 / 5)
    )

    equal_weight_bt = long_short_backtest(
        test_df,
        build_equal_weight_long_only_weights(test_df, rebalance_dates=test_rebalance_dates),
        target_col="target_5d_raw",
        transaction_cost=0.0005,
    )
    rolling5y_walk_forward_baseline_rows.append(
        rolling5y_metric_row(fold_name, "Equal-weight long-only universe", test_period, equal_weight_bt, periods_per_year=252 / 5)
    )

    random_bt = long_short_backtest(
        test_df,
        build_random_long_short_weights(test_df, rebalance_dates=test_rebalance_dates, seed=42),
        target_col="target_5d_raw",
        transaction_cost=0.0005,
    )
    rolling5y_walk_forward_baseline_rows.append(
        rolling5y_metric_row(fold_name, "Random dollar-neutral long-short", test_period, random_bt, periods_per_year=252 / 5)
    )

    momentum_bt = long_short_backtest(
        test_df,
        build_long_short_weights(test_df, "return_20", rebalance_dates=test_rebalance_dates),
        target_col="target_5d_raw",
        transaction_cost=0.0005,
    )
    rolling5y_walk_forward_baseline_rows.append(
        rolling5y_metric_row(fold_name, "20-day momentum long-short", test_period, momentum_bt, periods_per_year=252 / 5)
    )

    if "prepare_robust_simple_alpha_signals" in globals() and "run_robust_simple_alpha_search" in globals() and "robust_simple_backtest" in globals():
        val_prepared, signal_cols = prepare_robust_simple_alpha_signals(val_df)
        test_prepared, _ = prepare_robust_simple_alpha_signals(test_df)
        if signal_cols:
            simple_search_results = run_robust_simple_alpha_search(val_prepared, signal_cols)
            if not simple_search_results.empty:
                best_simple_config = simple_search_results.iloc[0]
                simple_bt, _, simple_net_metrics = robust_simple_backtest(
                    test_prepared,
                    best_simple_config["Signal"],
                    direction=int(best_simple_config["Direction"]),
                    top_frac=float(best_simple_config["top_frac"]),
                    rebalance_days=int(best_simple_config["rebalance_days"]),
                    weighting_method=best_simple_config["weighting_method"],
                    transaction_cost=0.0005,
                )
                rolling5y_walk_forward_baseline_rows.append({
                    "Fold": fold_name,
                    "Strategy": "Best simple alpha long-short",
                    "Test Period": test_period,
                    "Test Net Cumulative Return": simple_net_metrics["Cumulative Return"],
                    "Test Sharpe": simple_net_metrics["Sharpe Ratio"],
                    "Test Max Drawdown": simple_net_metrics["Max Drawdown"],
                    "Test Average Turnover": simple_net_metrics["Average Turnover"],
                    "Number of Rebalance Periods": simple_net_metrics["Number of Rebalance Periods"],
                })

rolling5y_walk_forward_results_df = pd.DataFrame(rolling5y_walk_forward_rows)
rolling5y_walk_forward_baseline_comparison_df = pd.DataFrame(rolling5y_walk_forward_baseline_rows)
rolling5y_walk_forward_baseline_comparison_df["Test Year"] = rolling5y_walk_forward_baseline_comparison_df["Test Period"].str.slice(0, 4).astype(int)

rolling5y_yearly_return_comparison_df = (
    rolling5y_walk_forward_baseline_comparison_df
    .pivot(index="Test Year", columns="Strategy", values="Test Net Cumulative Return")
    .reset_index()
    .sort_values("Test Year")
    .reset_index(drop=True)
)
rolling5y_yearly_return_comparison_df.columns.name = None

rolling5y_yearly_sharpe_comparison_df = (
    rolling5y_walk_forward_baseline_comparison_df
    .pivot(index="Test Year", columns="Strategy", values="Test Sharpe")
    .reset_index()
    .sort_values("Test Year")
    .reset_index(drop=True)
)
rolling5y_yearly_sharpe_comparison_df.columns.name = None

rolling5y_walk_forward_baseline_summary_df = (
    rolling5y_walk_forward_baseline_comparison_df
    .groupby("Strategy", as_index=False)
    .agg(
        **{
            "Average Test Net Cumulative Return": ("Test Net Cumulative Return", "mean"),
            "Average Test Sharpe": ("Test Sharpe", "mean"),
            "Average Test Max Drawdown": ("Test Max Drawdown", "mean"),
            "Average Test Average Turnover": ("Test Average Turnover", "mean"),
            "Number of positive-return folds": ("Test Net Cumulative Return", lambda x: (x > 0).sum()),
        }
    )
    .sort_values("Average Test Sharpe", ascending=False)
    .reset_index(drop=True)
)

display(rolling5y_walk_forward_results_df)
display(rolling5y_yearly_return_comparison_df)
display(rolling5y_yearly_sharpe_comparison_df)
display(rolling5y_walk_forward_baseline_summary_df)


Fold 1 | train 2010-01-01 to 2014-12-31 rows=43,411 stocks=44 | validation 2015-01-01 to 2015-12-31 rows=11,075 stocks=44 | test 2016-01-01 to 2016-12-31 rows=11,383 stocks=46
Fold 1 selected candidate=slightly stronger direction=1 test_return=0.0011 test_sharpe=0.0106
Fold 2 | train 2011-01-01 to 2015-12-31 rows=54,486 stocks=44 | validation 2016-01-01 to 2016-12-31 rows=11,383 stocks=46 | test 2017-01-01 to 2017-12-31 rows=11,640 stocks=47
Fold 2 selected candidate=conservative shallow direction=1 test_return=0.2707 test_sharpe=3.0825
Fold 3 | train 2012-01-01 to 2016-12-31 rows=55,199 stocks=46 | validation 2017-01-01 to 2017-12-31 rows=11,640 stocks=47 | test 2018-01-01 to 2018-12-31 rows=11,795 stocks=47
Fold 3 selected candidate=slightly stronger direction=1 test_return=-0.0391 test_sharpe=-0.5649
Fold 4 | train 2013-01-01 to 2017-12-31 rows=56,143 stocks=47 | validation 2018-01-01 to 2018-12-31 rows=11,795 stocks=47 | test 2019-01-01 to 2019-12-31 rows=11,843 stocks=47
Fold 4 se

,Fold,Train Period,Validation Period,Test Period,Number of Train Rows,Number of Validation Rows,Number of Test Rows,Number of Train Stocks,Number of Validation Stocks,Number of Test Stocks,Selected Candidate,Selected Params,Selected Direction,Validation Net Sharpe,Validation Daily Rank IC Mean,Test Spearman Rank IC,Test Daily Rank IC Mean,Test Net Cumulative Return,Test Sharpe,Test Max Drawdown,Test Average Turnover,Number of Rebalance Periods
0,Fold 1,2010-01-01 to 2014-12-31,2015-01-01 to 2015-12-31,2016-01-01 to 2016-12-31,43411,11075,11383,44,44,46,slightly stronger,"{'n_estimators': 300, 'learning_rate': 0.02, '...",1,0.297129,0.011559,0.004898,-0.002604,0.001059,0.010605,-0.066732,0.808279,51
1,Fold 2,2011-01-01 to 2015-12-31,2016-01-01 to 2016-12-31,2017-01-01 to 2017-12-31,54486,11383,11640,44,46,47,conservative shallow,"{'n_estimators': 100, 'learning_rate': 0.03, '...",1,0.169895,0.006872,0.048795,0.054713,0.270700,3.082538,-0.030625,0.793028,51
2,Fold 3,2012-01-01 to 2016-12-31,2017-01-01 to 2017-12-31,2018-01-01 to 2018-12-31,55199,11640,11795,46,47,47,slightly stronger,"{'n_estimators': 300, 'learning_rate': 0.02, '...",1,2.812582,0.037672,-0.019272,-0.014691,-0.039104,-0.564883,-0.065294,0.895425,51
3,Fold 4,2013-01-01 to 2017-12-31,2018-01-01 to 2018-12-31,2019-01-01 to 2019-12-31,56143,11795,11843,47,47,47,conservative shallow,"{'n_estimators': 100, 'learning_rate': 0.03, '...",-1,0.262611,-0.000981,-0.051576,-0.046678,-0.165906,-2.158617,-0.199376,0.773420,51
4,Fold 5,2014-01-01 to 2018-12-31,2019-01-01 to 2019-12-31,2020-01-01 to 2020-12-31,56975,11843,11891,47,47,47,slightly stronger,"{'n_estimators': 300, 'learning_rate': 0.02, '...",1,1.604140,0.032047,0.027755,0.019199,0.089016,1.025036,-0.059519,0.930283,51
5,Fold 6,2015-01-01 to 2019-12-31,2020-01-01 to 2020-12-31,2021-01-01 to 2021-12-31,57736,11891,11888,47,47,48,slightly stronger,"{'n_estimators': 300, 'learning_rate': 0.02, '...",1,1.016648,0.038650,0.026202,0.022593,0.120134,1.096970,-0.087917,0.906318,51
6,Fold 7,2016-01-01 to 2020-12-31,2021-01-01 to 2021-12-31,2022-01-01 to 2022-12-31,58552,11888,12092,47,48,49,slightly stronger,"{'n_estimators': 300, 'learning_rate': 0.02, '...",1,2.412705,0.043287,0.045994,0.035735,0.166175,1.647892,-0.053267,0.895425,51
7,Fold 8,2017-01-01 to 2021-12-31,2022-01-01 to 2022-12-31,2023-01-01 to 2023-12-31,59057,12092,12250,48,49,49,medium,"{'n_estimators': 200, 'learning_rate': 0.03, '...",1,1.716932,0.039550,-0.029431,-0.019109,-0.103947,-1.092665,-0.156214,0.777778,50
8,Fold 9,2018-01-01 to 2022-12-31,2023-01-01 to 2023-12-31,2024-01-01 to 2024-12-31,59509,12250,12054,49,49,49,slightly stronger,"{'n_estimators': 300, 'learning_rate': 0.02, '...",-1,0.400128,0.019522,0.039521,0.033826,0.105481,0.724358,-0.111507,0.777778,50


,Test Year,20-day momentum long-short,Best simple alpha long-short,Equal-weight long-only universe,LightGBM long-short,Random dollar-neutral long-short
0,2016,-0.074493,0.105937,0.430319,0.001059,-0.128351
1,2017,-0.059143,-0.011974,0.498807,0.270700,-0.131865
2,2018,0.060190,-0.203595,-0.129048,-0.039104,-0.034494
3,2019,-0.183776,-0.076458,0.683637,-0.165906,0.060642
4,2020,0.016581,0.010613,0.563906,0.089016,-0.029485
5,2021,-0.097245,0.016698,0.516596,0.120134,0.020987
6,2022,-0.148060,0.100733,-0.234447,0.166175,-0.067682
7,2023,0.089952,-0.103959,0.535165,-0.103947,-0.122185
8,2024,-0.030355,0.049727,0.122669,0.105481,0.092180


,Test Year,20-day momentum long-short,Best simple alpha long-short,Equal-weight long-only universe,LightGBM long-short,Random dollar-neutral long-short
0,2016,-0.810739,1.062040,2.162522,0.010605,-1.457498
1,2017,-0.912252,-0.234563,3.461683,3.082538,-1.960605
2,2018,0.651540,-1.894026,-0.486459,-0.564883,-0.450228
3,2019,-2.852073,-1.627406,3.054004,-2.158617,0.847943
4,2020,0.135599,0.064575,1.451777,1.025036,-0.407016
5,2021,-0.989714,0.176894,2.162595,1.096970,0.294031
6,2022,-1.557431,1.897483,-0.638338,1.647892,-1.069106
7,2023,0.800852,-1.221244,2.148518,-1.092665,-1.371430
8,2024,-0.223218,0.529908,0.379235,0.724358,0.904371


,Strategy,Average Test Net Cumulative Return,Average Test Sharpe,Average Test Max Drawdown,Average Test Average Turnover,Number of positive-return folds
0,Equal-weight long-only universe,0.331956,1.521726,-0.187468,0.020829,7
1,LightGBM long-short,0.049290,0.419026,-0.092273,0.839748,6
2,Best simple alpha long-short,-0.012475,-0.138482,-0.089012,1.166176,5
3,Random dollar-neutral long-short,-0.037806,-0.518838,-0.090014,1.614999,3
4,20-day momentum long-short,-0.047372,-0.639715,-0.122950,0.809024,3


In [ ]:
# Rerun only Fold 4 / Test Year 2019 with fixed direction = 1

fold4_spec = rolling5y_fold_specs[3]
fold_name, train_start, train_end, val_start, val_end, test_start, test_end = fold4_spec

split, _ = create_time_split(
    rolling5y_source_df,
    train_start=train_start,
    train_end=train_end,
    val_start=val_start,
    val_end=val_end,
    test_start=test_start,
    test_end=test_end,
)

train_df = split["train"].copy()
val_df = split["validation"].copy()
test_df = split["test"].copy()

X_train = train_df[rolling5y_feature_cols]
y_train = train_df["target_5d_cs"]

X_val = val_df[rolling5y_feature_cols]
y_val = val_df["target_5d_cs"]

X_test = test_df[rolling5y_feature_cols]
y_test = test_df["target_5d_cs"]

# 1. Validation only selects candidate, fixed direction = 1
candidate_rows = []

for candidate in rolling5y_lgbm_candidates:
    model = rolling5y_build_model(candidate["params"])
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)   # fixed direction = 1

    tmp = val_df[["date", "code", "target_5d_cs", "target_5d_raw"]].copy()
    tmp["rolling5y_signal"] = val_pred

    pred_metrics = evaluate_prediction(y_val, val_pred)
    daily_ic_mean, _, _ = daily_rank_ic_summary(
        tmp, "rolling5y_signal", target_col="target_5d_cs"
    )

    rebalance_dates = get_rebalance_dates(tmp, every_n_days=5)
    weights = build_long_short_weights(
        tmp, "rolling5y_signal", rebalance_dates=rebalance_dates
    )

    bt = long_short_backtest(
        tmp,
        weights,
        target_col="target_5d_raw",
        transaction_cost=0.0005,
    )

    net_metrics = backtest_metrics(bt, return_col="net_return")

    candidate_rows.append({
        "Candidate": candidate["Candidate"],
        "Params": candidate["params"],
        "Direction": 1,
        "Validation Net Sharpe": net_metrics["Sharpe Ratio"],
        "Validation Daily Rank IC Mean": daily_ic_mean,
        "Validation Net Cumulative Return": net_metrics["Cumulative Return"],
        "Validation Max Drawdown": net_metrics["Max Drawdown"],
        "Validation Average Turnover": net_metrics["Average Turnover"],
        "Validation Number of Rebalance Periods": net_metrics["Number of Rebalance Periods"],
        "Validation Spearman Rank IC": pred_metrics["Spearman Rank IC"],
    })

fold4_candidate_results_fixed_dir_df = pd.DataFrame(candidate_rows)
selected = rolling5y_select_best_candidate(fold4_candidate_results_fixed_dir_df)

print("Fold 4 fixed direction=1 validation candidates:")
display(fold4_candidate_results_fixed_dir_df)

print("Selected candidate:")
display(pd.DataFrame([selected]))

# 2. Refit selected model on train + validation
final_model = rolling5y_build_model(selected["Params"])

X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

final_model.fit(X_train_val, y_train_val)

# 3. Test on 2019 with fixed direction = 1
test_signal = final_model.predict(X_test)

test_eval = test_df[["date", "code", "target_5d_cs", "target_5d_raw"]].copy()
test_eval["rolling5y_lgbm_signal"] = test_signal

test_pred_metrics = evaluate_prediction(y_test, test_signal)

test_daily_ic_mean, _, _ = daily_rank_ic_summary(
    test_eval,
    "rolling5y_lgbm_signal",
    target_col="target_5d_cs",
)

test_rebalance_dates = get_rebalance_dates(test_eval, every_n_days=5)

test_weights = build_long_short_weights(
    test_eval,
    "rolling5y_lgbm_signal",
    rebalance_dates=test_rebalance_dates,
)

fold4_lgbm_bt_fixed_dir = long_short_backtest(
    test_eval,
    test_weights,
    target_col="target_5d_raw",
    transaction_cost=0.0005,
)

test_metrics = backtest_metrics(
    fold4_lgbm_bt_fixed_dir,
    return_col="net_return",
    periods_per_year=252 / 5,
)

fold4_fixed_dir_result = {
    "Fold": fold_name,
    "Train Period": f"{train_start} to {train_end}",
    "Validation Period": f"{val_start} to {val_end}",
    "Test Period": f"{test_start} to {test_end}",
    "Selected Candidate": selected["Candidate"],
    "Selected Direction": 1,
    "Validation Net Sharpe": selected["Validation Net Sharpe"],
    "Validation Daily Rank IC Mean": selected["Validation Daily Rank IC Mean"],
    "Test Spearman Rank IC": test_pred_metrics["Spearman Rank IC"],
    "Test Daily Rank IC Mean": test_daily_ic_mean,
    "Test Net Cumulative Return": test_metrics["Cumulative Return"],
    "Test Sharpe": test_metrics["Sharpe Ratio"],
    "Test Max Drawdown": test_metrics["Max Drawdown"],
    "Test Average Turnover": test_metrics["Average Turnover"],
    "Number of Rebalance Periods": test_metrics["Number of Rebalance Periods"],
}

fold4_fixed_dir_result_df = pd.DataFrame([fold4_fixed_dir_result])

print(
    f"{fold_name} fixed direction=1 | "
    f"selected candidate={selected['Candidate']} | "
    f"test_return={test_metrics['Cumulative Return']:.4f} | "
    f"test_sharpe={test_metrics['Sharpe Ratio']:.4f}"
)

display(fold4_fixed_dir_result_df)

Fold 4 fixed direction=1 validation candidates:


,Candidate,Params,Direction,Validation Net Sharpe,Validation Daily Rank IC Mean,Validation Net Cumulative Return,Validation Max Drawdown,Validation Average Turnover,Validation Number of Rebalance Periods,Validation Spearman Rank IC
0,conservative shallow,"{'n_estimators': 100, 'learning_rate': 0.03, '...",1,-0.891036,0.000981,-0.067464,-0.115721,0.851852,51,0.000920
1,medium,"{'n_estimators': 200, 'learning_rate': 0.03, '...",1,0.070000,-0.006027,0.004993,-0.053146,0.923747,51,-0.004755
2,slightly stronger,"{'n_estimators': 300, 'learning_rate': 0.02, '...",1,0.134437,-0.003118,0.011495,-0.077123,0.915033,51,0.001591


Selected candidate:


,Candidate,Params,Direction,Validation Net Sharpe,Validation Daily Rank IC Mean,Validation Net Cumulative Return,Validation Max Drawdown,Validation Average Turnover,Validation Number of Rebalance Periods,Validation Spearman Rank IC,Validation Average Turnover Sort,Validation Max Drawdown Abs Sort,Validation Net Sharpe Sort,Validation Net Cumulative Return Sort
2,slightly stronger,"{'n_estimators': 300, 'learning_rate': 0.02, '...",1,0.134437,-0.003118,0.011495,-0.077123,0.915033,51,0.001591,0.915033,0.077123,0.134437,0.011495


Fold 4 fixed direction=1 | selected candidate=slightly stronger | test_return=0.1406 | test_sharpe=1.7197


,Fold,Train Period,Validation Period,Test Period,Selected Candidate,Selected Direction,Validation Net Sharpe,Validation Daily Rank IC Mean,Test Spearman Rank IC,Test Daily Rank IC Mean,Test Net Cumulative Return,Test Sharpe,Test Max Drawdown,Test Average Turnover,Number of Rebalance Periods
0,Fold 4,2013-01-01 to 2017-12-31,2018-01-01 to 2018-12-31,2019-01-01 to 2019-12-31,slightly stronger,1,0.134437,-0.003118,0.051149,0.04498,0.140605,1.719657,-0.035335,0.840959,51


In [ ]:
# Rerun only Fold 9 / Test Year 2024 with fixed direction = 1

fold9_spec = rolling5y_fold_specs[8]
fold_name, train_start, train_end, val_start, val_end, test_start, test_end = fold9_spec

split, _ = create_time_split(
    rolling5y_source_df,
    train_start=train_start,
    train_end=train_end,
    val_start=val_start,
    val_end=val_end,
    test_start=test_start,
    test_end=test_end,
)

train_df = split["train"].copy()
val_df = split["validation"].copy()
test_df = split["test"].copy()

X_train = train_df[rolling5y_feature_cols]
y_train = train_df["target_5d_cs"]

X_val = val_df[rolling5y_feature_cols]
y_val = val_df["target_5d_cs"]

X_test = test_df[rolling5y_feature_cols]
y_test = test_df["target_5d_cs"]

# 1. Validation only selects candidate, fixed direction = 1
candidate_rows = []

for candidate in rolling5y_lgbm_candidates:
    model = rolling5y_build_model(candidate["params"])
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)  # fixed direction = 1

    tmp = val_df[["date", "code", "target_5d_cs", "target_5d_raw"]].copy()
    tmp["rolling5y_signal"] = val_pred

    pred_metrics = evaluate_prediction(y_val, val_pred)
    daily_ic_mean, _, _ = daily_rank_ic_summary(
        tmp,
        "rolling5y_signal",
        target_col="target_5d_cs",
    )

    rebalance_dates = get_rebalance_dates(tmp, every_n_days=5)
    weights = build_long_short_weights(
        tmp,
        "rolling5y_signal",
        rebalance_dates=rebalance_dates,
    )

    bt = long_short_backtest(
        tmp,
        weights,
        target_col="target_5d_raw",
        transaction_cost=0.0005,
    )

    net_metrics = backtest_metrics(bt, return_col="net_return")

    candidate_rows.append({
        "Candidate": candidate["Candidate"],
        "Params": candidate["params"],
        "Direction": 1,
        "Validation Net Sharpe": net_metrics["Sharpe Ratio"],
        "Validation Daily Rank IC Mean": daily_ic_mean,
        "Validation Net Cumulative Return": net_metrics["Cumulative Return"],
        "Validation Max Drawdown": net_metrics["Max Drawdown"],
        "Validation Average Turnover": net_metrics["Average Turnover"],
        "Validation Number of Rebalance Periods": net_metrics["Number of Rebalance Periods"],
        "Validation Spearman Rank IC": pred_metrics["Spearman Rank IC"],
    })

fold9_candidate_results_fixed_dir_df = pd.DataFrame(candidate_rows)
selected = rolling5y_select_best_candidate(fold9_candidate_results_fixed_dir_df)

print("Fold 9 fixed direction=1 validation candidates:")
display(fold9_candidate_results_fixed_dir_df)

print("Selected candidate:")
display(pd.DataFrame([selected]))

# 2. Refit selected model on train + validation
final_model = rolling5y_build_model(selected["Params"])

X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

final_model.fit(X_train_val, y_train_val)

# 3. Test on 2024 with fixed direction = 1
test_signal = final_model.predict(X_test)

test_eval = test_df[["date", "code", "target_5d_cs", "target_5d_raw"]].copy()
test_eval["rolling5y_lgbm_signal"] = test_signal

test_pred_metrics = evaluate_prediction(y_test, test_signal)

test_daily_ic_mean, _, _ = daily_rank_ic_summary(
    test_eval,
    "rolling5y_lgbm_signal",
    target_col="target_5d_cs",
)

test_rebalance_dates = get_rebalance_dates(test_eval, every_n_days=5)

test_weights = build_long_short_weights(
    test_eval,
    "rolling5y_lgbm_signal",
    rebalance_dates=test_rebalance_dates,
)

fold9_lgbm_bt_fixed_dir = long_short_backtest(
    test_eval,
    test_weights,
    target_col="target_5d_raw",
    transaction_cost=0.0005,
)

test_metrics = backtest_metrics(
    fold9_lgbm_bt_fixed_dir,
    return_col="net_return",
    periods_per_year=252 / 5,
)

fold9_fixed_dir_result = {
    "Fold": fold_name,
    "Train Period": f"{train_start} to {train_end}",
    "Validation Period": f"{val_start} to {val_end}",
    "Test Period": f"{test_start} to {test_end}",
    "Selected Candidate": selected["Candidate"],
    "Selected Direction": 1,
    "Validation Net Sharpe": selected["Validation Net Sharpe"],
    "Validation Daily Rank IC Mean": selected["Validation Daily Rank IC Mean"],
    "Test Spearman Rank IC": test_pred_metrics["Spearman Rank IC"],
    "Test Daily Rank IC Mean": test_daily_ic_mean,
    "Test Net Cumulative Return": test_metrics["Cumulative Return"],
    "Test Sharpe": test_metrics["Sharpe Ratio"],
    "Test Max Drawdown": test_metrics["Max Drawdown"],
    "Test Average Turnover": test_metrics["Average Turnover"],
    "Number of Rebalance Periods": test_metrics["Number of Rebalance Periods"],
}

fold9_fixed_dir_result_df = pd.DataFrame([fold9_fixed_dir_result])

print(
    f"{fold_name} fixed direction=1 | "
    f"selected candidate={selected['Candidate']} | "
    f"test_return={test_metrics['Cumulative Return']:.4f} | "
    f"test_sharpe={test_metrics['Sharpe Ratio']:.4f}"
)

display(fold9_fixed_dir_result_df)

Fold 9 fixed direction=1 validation candidates:


,Candidate,Params,Direction,Validation Net Sharpe,Validation Daily Rank IC Mean,Validation Net Cumulative Return,Validation Max Drawdown,Validation Average Turnover,Validation Number of Rebalance Periods,Validation Spearman Rank IC
0,conservative shallow,"{'n_estimators': 100, 'learning_rate': 0.03, '...",1,-0.523843,-0.008437,-0.046155,-0.099751,0.686667,50,-0.021004
1,medium,"{'n_estimators': 200, 'learning_rate': 0.03, '...",1,0.072155,-0.002260,0.006656,-0.088723,0.782222,50,-0.015781
2,slightly stronger,"{'n_estimators': 300, 'learning_rate': 0.02, '...",1,-0.898786,-0.019522,-0.075409,-0.115266,0.766667,50,-0.028611


Selected candidate:


,Candidate,Params,Direction,Validation Net Sharpe,Validation Daily Rank IC Mean,Validation Net Cumulative Return,Validation Max Drawdown,Validation Average Turnover,Validation Number of Rebalance Periods,Validation Spearman Rank IC,Validation Average Turnover Sort,Validation Max Drawdown Abs Sort,Validation Net Sharpe Sort,Validation Net Cumulative Return Sort
1,medium,"{'n_estimators': 200, 'learning_rate': 0.03, '...",1,0.072155,-0.00226,0.006656,-0.088723,0.782222,50,-0.015781,0.782222,0.088723,0.072155,0.006656


Fold 9 fixed direction=1 | selected candidate=medium | test_return=-0.1402 | test_sharpe=-1.0634


,Fold,Train Period,Validation Period,Test Period,Selected Candidate,Selected Direction,Validation Net Sharpe,Validation Daily Rank IC Mean,Test Spearman Rank IC,Test Daily Rank IC Mean,Test Net Cumulative Return,Test Sharpe,Test Max Drawdown,Test Average Turnover,Number of Rebalance Periods
0,Fold 9,2018-01-01 to 2022-12-31,2023-01-01 to 2023-12-31,2024-01-01 to 2024-12-31,medium,1,0.072155,-0.00226,-0.039439,-0.03875,-0.140243,-1.063413,-0.190892,0.793333,50
